# TripAdvisor Annotated Dataset Inspection

This notebook examines the English Trip-MA hotel review corpus selected as the primary new dataset for the C-Top2Vec extension. The purpose of this stage is to verify the dataset files, inspect the annotation structure, and summarise the available review and aspect information before preprocessing and constructing controlled multi-aspect documents.

## 1. Dataset Location and File Verification

The Trip-MA English dataset is stored locally within the project directory. Before loading the annotations, the paths to the training and test files are defined and checked to confirm that both files are accessible from the current Jupyter environment.

In [3]:
from pathlib import Path

data_dir = (
    Path.home()
    / "Documents"
    / "contextual-top2vec-reproduction"
    / "data"
    / "trip-maml"
    / "EN"
)

train_path = data_dir / "english_training.json"
test_path = data_dir / "english_test.json"

print("Training file exists:", train_path.exists())
print("Test file exists:", test_path.exists())

Training file exists: True
Test file exists: True


## 2. Loading the Annotation Files

The Trip-MA files use a line-delimited JSON format rather than a single JSON array. Each non-empty line contains one complete JSON object. The files are therefore loaded line by line so that each record can be parsed independently.

In [4]:
import json

def load_jsonl(path):
    records = []

    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()

            if not line:
                continue

            records.append(json.loads(line))

    return records


train_data = load_jsonl(train_path)
test_data = load_jsonl(test_path)

print("Training records:", len(train_data))
print("Test records:", len(test_data))
print("Total records:", len(train_data) + len(test_data))

Training records: 258
Test records: 111
Total records: 369


## 3. Initial Inspection of Dataset Structure

Before converting the dataset into a tabular format, the first training record is inspected to identify the available fields and understand how review text and aspect annotations are represented.

In [5]:
print(train_data[0])

{'annotatorId': 2, 'segmentLabels': [{'OTHER': 'p'}, {'NOTRELATED': 'x'}, {'OTHER': 'p'}, {'SERVICE': 'p'}, {'ROOMS': 'p'}, {'CLEANLINESS': 'p', 'ROOMS': 'p'}, {'FOOD': 'p'}, {'FOOD': 'p'}, {'LOCATION': 'ip'}, {'OTHER': 'p'}, {'OTHER': 'p'}, {'OTHER': 'p'}], 'ratingOverall': 5, 'ratingRoom': 5, 'author': 'travellerseattle', 'hotelId': '277882', 'reviewId': '277882:54', 'ratingLocation': 4, 'ratingService': 5, 'ratingBusiness': 3, 'ratingValue': 5, 'ratingCleanliness': 5, 'date': 'May 15, 2007', 'ratingCheckin': 5, 'segments': ['LOVED/v/LOVED THE//THE HAMPTON/n/HAMPTON INN/n/INN SEAPORT/n/SEAPORT !//! !//! !//! !//! !//! !//! !//! !//! !//! !//! !//! !//! !//! !//! !//! !//! !//! !//!', 'Just/r/Just returned/v/return from//from a//a 3//3 night/n/night stay/n/stay .//.', 'This//This is/v/be a//a FABULOUS/a/FABULOUS hotel/n/hotel .//.', 'The//The front/n/front desk/n/desk staff/n/staff ,//, the//the doormen/n/doorman ,//, the//the breakfast/n/breakfast staff/n/staff ,//, EVERYONE/n/EVERYO

## 4. Inspection of Annotation Structure

Each review contains a list of sentence-level segments and a corresponding list of aspect annotations. The position of each annotation entry corresponds to the same position in the segment list, allowing each sentence to be linked directly to its annotated aspects.

The annotations are multi-label, meaning that a single sentence may be associated with more than one aspect. This is particularly relevant to the current project because C-Top2Vec is intended to identify multiple topical regions within documents.

Before preprocessing the annotations, the aspect categories and annotation codes present in the corpus are examined empirically.

In [7]:
from collections import Counter

all_records = train_data + test_data

aspect_counts = Counter()
annotation_code_counts = Counter()

for record in all_records:
    for label_dict in record["segmentLabels"]:
        for aspect, code in label_dict.items():
            aspect_counts[aspect] += 1
            annotation_code_counts[code] += 1

print("Aspect categories:")
for aspect, count in sorted(aspect_counts.items()):
    print(f"{aspect}: {count}")

print("\nAnnotation codes:")
for code, count in sorted(annotation_code_counts.items()):
    print(f"{code}: {count}")

Aspect categories:
BUILDING: 226
BUSINESS: 24
CHECKIN: 142
CLEANLINESS: 211
FOOD: 401
LOCATION: 451
NOTRELATED: 376
OTHER: 1188
ROOMS: 751
SERVICE: 681
VALUE: 257

Annotation codes:
in: 464
ip: 511
ix: 25
n: 656
p: 2343
x: 709


## 5. Verification of Segment-Annotation Alignment

Because the dataset will later be used to construct controlled multi-aspect documents, it is important to confirm that every sentence segment has a corresponding annotation entry. Misaligned segments and annotations would make it impossible to retain reliable aspect boundaries during dataset construction.

In [8]:
alignment_issues = []

for i, record in enumerate(all_records):
    n_segments = len(record["segments"])
    n_labels = len(record["segmentLabels"])

    if n_segments != n_labels:
        alignment_issues.append(
            {
                "record_index": i,
                "reviewId": record.get("reviewId"),
                "segments": n_segments,
                "labels": n_labels
            }
        )

print("Total records checked:", len(all_records))
print("Records with alignment issues:", len(alignment_issues))

if alignment_issues:
    print("\nFirst few issues:")
    print(alignment_issues[:5])
else:
    print("All segment lists align with their annotation lists.")

Total records checked: 369
Records with alignment issues: 0
All segment lists align with their annotation lists.


## 6. Corpus Summary

The loaded training and test files contain 369 unique English hotel reviews. This is consistent with the Trip-MA annotation design. The complete annotated English corpus contains 442 reviews, of which 73 were independently annotated by all three annotators for inter-annotator agreement. The remaining 369 unique reviews were divided into training and test sets.

Inspection of the loaded corpus identified all eleven aspect categories: BUILDING, BUSINESS, CHECKIN, CLEANLINESS, FOOD, LOCATION, NOTRELATED, OTHER, ROOMS, SERVICE, and VALUE.

All 369 reviews were also checked for alignment between sentence segments and their corresponding annotation entries. No alignment problems were identified, confirming that the segment-level annotations can be reliably linked to the review text for subsequent preprocessing and controlled dataset construction.

In [10]:
import pandas as pd

rows = []

for split_name, records in [
    ("train", train_data),
    ("test", test_data)
]:
    for record in records:

        for segment_index, (segment, labels) in enumerate(
            zip(record["segments"], record["segmentLabels"])
        ):

            rows.append({
                "split": split_name,
                "review_id": record["reviewId"],
                "hotel_id": record["hotelId"],
                "annotator_id": record["annotatorId"],
                "segment_index": segment_index,
                "segment_raw": segment,
                "aspect_labels": list(labels.keys()),
                "annotation_codes": labels,
                "n_aspects": len(labels),
                "rating_overall": record["ratingOverall"]
            })

segments_df = pd.DataFrame(rows)

print("Sentence-level rows:", len(segments_df))
print("Unique reviews:", segments_df["review_id"].nunique())
print("Maximum aspects on one segment:", segments_df["n_aspects"].max())

segments_df.head(10)

Sentence-level rows: 4856
Unique reviews: 369
Maximum aspects on one segment: 6


,split,review_id,hotel_id,annotator_id,segment_index,segment_raw,aspect_labels,annotation_codes,n_aspects,rating_overall
0,train,277882:54,277882,2,0,LOVED/v/LOVED THE//THE HAMPTON/n/HAMPTON INN/n...,[OTHER],{'OTHER': 'p'},1,5
1,train,277882:54,277882,2,1,Just/r/Just returned/v/return from//from a//a ...,[NOTRELATED],{'NOTRELATED': 'x'},1,5
2,train,277882:54,277882,2,2,This//This is/v/be a//a FABULOUS/a/FABULOUS ho...,[OTHER],{'OTHER': 'p'},1,5
3,train,277882:54,277882,2,3,The//The front/n/front desk/n/desk staff/n/sta...,[SERVICE],{'SERVICE': 'p'},1,5
4,train,277882:54,277882,2,4,The//The room/n/room was/v/be fabulous/a/fabul...,[ROOMS],{'ROOMS': 'p'},1,5
5,train,277882:54,277882,2,5,"Really/r/Really comfy/v/comfy beds/n/bed ,//, ...","[CLEANLINESS, ROOMS]","{'CLEANLINESS': 'p', 'ROOMS': 'p'}",2,5
6,train,277882:54,277882,2,6,The//The breakfasts/n/breakfast are/v/be great...,[FOOD],{'FOOD': 'p'},1,5
7,train,277882:54,277882,2,7,Good/n/Good coffee/n/coffee .//.,[FOOD],{'FOOD': 'p'},1,5
8,train,277882:54,277882,2,8,Just/r/Just around//around the//the corner/n/c...,[LOCATION],{'LOCATION': 'ip'},1,5
9,train,277882:54,277882,2,9,Highly/r/Highly recommend/v/recommend it//it .//.,[OTHER],{'OTHER': 'p'},1,5


### Sentence-Level Representation

The original nested JSON structure has been converted into a sentence-level table in which each row represents one annotated review segment. Review identifiers and segment order are retained so that the original document structure can be reconstructed later.

The `aspect_labels` field preserves all aspects assigned to each segment rather than forcing the annotation into a single class. This is important because some sentences contain more than one annotated aspect, which provides useful examples for examining C-Top2Vec's behaviour on multi-aspect text.

## 7. Reconstruction of Readable Segment Text

The segment text in the original dataset contains token-level linguistic annotations. Each token includes its observed word form together with additional part-of-speech and lemma information, separated by forward slashes.

For the C-Top2Vec extension, the surface text is reconstructed by retaining the original word form while removing the embedded linguistic annotations. The original `segment_raw` field is preserved so that preprocessing remains traceable and reproducible.

Only the text representation is cleaned at this stage. Review identifiers, segment order, aspect labels, and annotation codes remain unchanged.

In [11]:
import re

def reconstruct_text(segment):
    """
    Reconstruct readable text from the annotated Trip-MA token format.

    Example:
    'The//The room/n/room was/v/be great/a/great .//.'
    -> 'The room was great.'
    """

    surface_tokens = []

    for token in segment.split():
        # Keep the surface form before the first "/"
        surface = token.split("/", 1)[0]

        if surface:
            surface_tokens.append(surface)

    text = " ".join(surface_tokens)

    # Remove spaces before punctuation
    text = re.sub(r"\s+([.,!?;:%)\]])", r"\1", text)

    # Remove spaces immediately after opening brackets
    text = re.sub(r"([(\[])\s+", r"\1", text)

    # Reconnect common English contractions
    text = re.sub(r"\s+n['’]t\b", "n't", text)
    text = re.sub(r"\s+['’](s|re|ve|ll|d|m)\b", r"'\1", text)

    # Collapse accidental repeated spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


segments_df["segment_text"] = segments_df["segment_raw"].apply(reconstruct_text)

segments_df[
    [
        "review_id",
        "segment_index",
        "segment_text",
        "aspect_labels",
        "n_aspects"
    ]
].head(10)

,review_id,segment_index,segment_text,aspect_labels,n_aspects
0,277882:54,0,LOVED THE HAMPTON INN SEAPORT!!!!!!!!!!!!!!!!!!,[OTHER],1
1,277882:54,1,Just returned from a 3 night stay.,[NOTRELATED],1
2,277882:54,2,This is a FABULOUS hotel.,[OTHER],1
3,277882:54,3,"The front desk staff, the doormen, the breakfa...",[SERVICE],1
4,277882:54,4,The room was fabulous too.,[ROOMS],1
5,277882:54,5,"Really comfy beds, great decorating, and super...","[CLEANLINESS, ROOMS]",2
6,277882:54,6,"The breakfasts are great - fresh fruit, bagels...",[FOOD],1
7,277882:54,7,Good coffee.,[FOOD],1
8,277882:54,8,Just around the corner from the hotel is a FAB...,[LOCATION],1
9,277882:54,9,Highly recommend it.,[OTHER],1


### 7.1 Validation of Text Reconstruction

The reconstructed text is checked to ensure that preprocessing has not removed any segments or altered their alignment with the original annotations. A small sample is also displayed alongside the original representation for manual inspection.

In [12]:
print("Total raw segments:", segments_df["segment_raw"].notna().sum())
print("Total reconstructed segments:", segments_df["segment_text"].notna().sum())
print("Empty reconstructed segments:", (segments_df["segment_text"].str.strip() == "").sum())

display(
    segments_df[
        [
            "segment_raw",
            "segment_text",
            "aspect_labels"
        ]
    ].head(10)
)

Total raw segments: 4856
Total reconstructed segments: 4856
Empty reconstructed segments: 0


,segment_raw,segment_text,aspect_labels
0,LOVED/v/LOVED THE//THE HAMPTON/n/HAMPTON INN/n...,LOVED THE HAMPTON INN SEAPORT!!!!!!!!!!!!!!!!!!,[OTHER]
1,Just/r/Just returned/v/return from//from a//a ...,Just returned from a 3 night stay.,[NOTRELATED]
2,This//This is/v/be a//a FABULOUS/a/FABULOUS ho...,This is a FABULOUS hotel.,[OTHER]
3,The//The front/n/front desk/n/desk staff/n/sta...,"The front desk staff, the doormen, the breakfa...",[SERVICE]
4,The//The room/n/room was/v/be fabulous/a/fabul...,The room was fabulous too.,[ROOMS]
5,"Really/r/Really comfy/v/comfy beds/n/bed ,//, ...","Really comfy beds, great decorating, and super...","[CLEANLINESS, ROOMS]"
6,The//The breakfasts/n/breakfast are/v/be great...,"The breakfasts are great - fresh fruit, bagels...",[FOOD]
7,Good/n/Good coffee/n/coffee .//.,Good coffee.,[FOOD]
8,Just/r/Just around//around the//the corner/n/c...,Just around the corner from the hotel is a FAB...,[LOCATION]
9,Highly/r/Highly recommend/v/recommend it//it .//.,Highly recommend it.,[OTHER]


## 8. Segment Length Analysis

C-Top2Vec constructs contextual token representations using sliding windows over the input text. In the original study, a contextual window size of 50 tokens was used.

Because the TripAdvisor corpus is annotated at the sentence-segment level, the length of each segment is examined to determine whether individual annotated segments are generally shorter or longer than the 50-token contextual window. This analysis informs the later construction of controlled multi-aspect documents, where several annotated segments may need to be combined so that aspect transitions occur within meaningful contextual windows.

In [13]:
segments_df["token_count"] = (
    segments_df["segment_text"]
    .str.split()
    .str.len()
)

print("Number of segments:", len(segments_df))
print("Minimum tokens:", segments_df["token_count"].min())
print("Maximum tokens:", segments_df["token_count"].max())
print("Mean tokens:", round(segments_df["token_count"].mean(), 2))
print("Median tokens:", segments_df["token_count"].median())

print("\nSegments under 50 tokens:",
      (segments_df["token_count"] < 50).sum())

print("Segments exactly 50 tokens:",
      (segments_df["token_count"] == 50).sum())

print("Segments over 50 tokens:",
      (segments_df["token_count"] > 50).sum())

print(
    "\nPercentage under 50 tokens:",
    round(
        (segments_df["token_count"] < 50).mean() * 100,
        2
    ),
    "%"
)

Number of segments: 4856
Minimum tokens: 1
Maximum tokens: 216
Mean tokens: 15.42
Median tokens: 13.0

Segments under 50 tokens: 4786
Segments exactly 50 tokens: 8
Segments over 50 tokens: 62

Percentage under 50 tokens: 98.56 %


### 8.1 Length Distribution by Aspect

Segment length is also examined across the annotated aspect categories. Since a segment may contain multiple aspect labels, each segment is represented once for every aspect assigned to it. This allows the typical amount of text associated with each aspect to be compared.

In [14]:
aspect_length_rows = []

for _, row in segments_df.iterrows():
    for aspect in row["aspect_labels"]:
        aspect_length_rows.append({
            "aspect": aspect,
            "token_count": row["token_count"]
        })

aspect_length_df = pd.DataFrame(aspect_length_rows)

aspect_length_summary = (
    aspect_length_df
    .groupby("aspect")["token_count"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(2)
    .sort_values("count", ascending=False)
)

aspect_length_summary

,count,mean,median,min,max
aspect,,,,,
OTHER,1188,13.78,11.0,1,140
ROOMS,751,17.97,15.0,2,216
SERVICE,681,19.12,15.0,3,216
LOCATION,451,17.92,15.0,2,109
FOOD,401,16.62,14.0,1,216
NOTRELATED,376,18.08,15.0,1,140
VALUE,257,17.27,15.0,2,86
BUILDING,226,17.94,14.0,2,204
CLEANLINESS,211,17.91,13.0,1,140


### 8.2 Multi-Aspect Segment Lengths

Segments containing more than one annotated aspect are especially relevant to the extension experiment because they provide naturally occurring examples of multiple semantic aspects appearing within a small textual region.

In [15]:
multi_aspect_segments = segments_df[
    segments_df["n_aspects"] > 1
].copy()

print("Multi-aspect segments:", len(multi_aspect_segments))
print(
    "Percentage of all segments:",
    round(len(multi_aspect_segments) / len(segments_df) * 100, 2),
    "%"
)

print(
    "Mean token length:",
    round(multi_aspect_segments["token_count"].mean(), 2)
)

display(
    multi_aspect_segments[
        [
            "review_id",
            "segment_index",
            "segment_text",
            "aspect_labels",
            "token_count"
        ]
    ].head(10)
)

Multi-aspect segments: 556
Percentage of all segments: 11.45 %
Mean token length: 21.29


,review_id,segment_index,segment_text,aspect_labels,token_count
5,277882:54,5,"Really comfy beds, great decorating, and super...","[CLEANLINESS, ROOMS]",9
24,87076:21,2,Very nice reception.,"[BUILDING, CHECKIN]",3
31,268429:162,0,"Nice little hotel, friendly staff","[OTHER, SERVICE]",5
33,268429:162,2,The room was clean and well maintained.,"[CLEANLINESS, ROOMS]",7
52,301648:29,8,"The shuttle isn't regular enough, and the town...","[OTHER, NOTRELATED]",33
56,506158:339,2,We both work in the fashion industry and would...,"[BUSINESS, VALUE]",43
63,506158:339,9,The views of the city were great and it was so...,"[OTHER, LOCATION]",60
64,228634:67,0,loved the royal hotel is in excellent location on,"[OTHER, LOCATION]",9
66,228634:67,2,"room was attractive, clean, adequate size and ...","[CLEANLINESS, FOOD, ROOMS]",35
67,228634:67,3,Desk staff very helpful and pleasant---would d...,"[CHECKIN, OTHER]",10


## 9. Candidate Selection for Controlled Multi-Aspect Construction

The segment-length analysis shows that 98.56% of annotated segments contain fewer than 50 whitespace-delimited tokens, with a mean length of 15.42 tokens. This suggests that combining approximately three short annotated segments can create documents containing multiple aspect transitions within a textual span comparable to the 50-token contextual window used by C-Top2Vec.

For the controlled construction experiment, only segments with exactly one substantive aspect label are initially selected. Naturally multi-aspect segments are excluded from this construction stage because their internal aspect boundaries are not explicitly annotated. They are retained separately for later analysis as naturally occurring multi-aspect examples.

`OTHER` and `NOTRELATED` are also excluded because the constructed dataset is intended to contain clearly defined hotel aspects with known transitions.

To reduce extreme variation in constructed-document length, candidate segments are limited to between 5 and 25 whitespace-delimited tokens.

In [18]:
core_aspects = [
    "BUILDING",
    "BUSINESS",
    "CHECKIN",
    "CLEANLINESS",
    "FOOD",
    "LOCATION",
    "ROOMS",
    "SERVICE",
    "VALUE"
]

candidate_segments = segments_df[
    (segments_df["n_aspects"] == 1) &
    (segments_df["aspect_labels"].apply(
        lambda x: len(x) == 1 and x[0] in core_aspects
    )) &
    (segments_df["token_count"].between(5, 25))
].copy()

candidate_segments["aspect"] = candidate_segments["aspect_labels"].apply(
    lambda x: x[0]
)

print("Eligible candidate segments:", len(candidate_segments))
print("Unique source reviews:", candidate_segments["review_id"].nunique())

candidate_counts = (
    candidate_segments["aspect"]
    .value_counts()
    .rename_axis("aspect")
    .reset_index(name="candidate_segments")
)

candidate_counts

Eligible candidate segments: 1706
Unique source reviews: 339


,aspect,candidate_segments
0,ROOMS,448
1,SERVICE,374
2,FOOD,272
3,LOCATION,247
4,BUILDING,121
5,VALUE,117
6,CHECKIN,65
7,CLEANLINESS,49
8,BUSINESS,13


### 9.1 Unlabelled Segment Check

Some review segments may not contain any aspect annotation. These segments are identified before candidate selection so that the controlled construction dataset contains only clearly annotated examples.

In [19]:
empty_label_segments = segments_df[
    segments_df["aspect_labels"].apply(len) == 0
]

print("Segments with no aspect labels:", len(empty_label_segments))

display(
    empty_label_segments[
        [
            "review_id",
            "segment_index",
            "segment_text",
            "aspect_labels"
        ]
    ].head(10)
)

Segments with no aspect labels: 811


,review_id,segment_index,segment_text,aspect_labels
32,268429:162,1,My wife and I stayed at the Coral Princess Inn...,[]
41,80788:50,3,I must give them credit at least they left the...,[]
49,301648:29,5,Another guest was complaining about another is...,[]
69,238565:2,1,I decided to stay in,[]
75,238565:2,7,Luckily I managed to purchase earplugs later.,[]
78,223041:120,1,My friend and I stayed at the Elan in late Feb...,[]
91,192044:88,2,HotelConnect) mainly based on the many great r...,[]
107,247957:1088,1,"November 26, 2003My husband and I are two of A...",[]
110,247957:1088,4,Our decision on this property was made based o...,[]
111,247957:1088,5,The renovations to the buffet resulted in the ...,[]


## 10. Selection of Aspects for Controlled Construction

After filtering for single-aspect segments between 5 and 25 tokens, 1,706 eligible candidate segments remained across 339 source reviews.

The candidate distribution is uneven across aspects. In particular, the BUSINESS category contains only 13 eligible segments, substantially fewer than the other substantive aspects. To avoid excessive reuse or artificial oversampling of a very small category, BUSINESS is retained for descriptive analysis of the natural TripAdvisor corpus but excluded from the controlled constructed dataset.

The controlled experiment therefore uses eight aspects:

- BUILDING
- CHECKIN
- CLEANLINESS
- FOOD
- LOCATION
- ROOMS
- SERVICE
- VALUE

Unlabelled segments are also excluded because they do not provide a known aspect identity or boundary.

A fixed random seed is used throughout construction to make the process reproducible.

In [20]:
import pandas as pd
import numpy as np

RANDOM_SEED = 42

construction_aspects = [
    "BUILDING",
    "CHECKIN",
    "CLEANLINESS",
    "FOOD",
    "LOCATION",
    "ROOMS",
    "SERVICE",
    "VALUE"
]

construction_candidates = candidate_segments[
    candidate_segments["aspect"].isin(construction_aspects)
].copy()

print("Controlled-construction candidates:",
      len(construction_candidates))

print("Unique source reviews:",
      construction_candidates["review_id"].nunique())

print("\nCandidate counts:")
print(
    construction_candidates["aspect"]
    .value_counts()
    .reindex(construction_aspects)
)

Controlled-construction candidates: 1693
Unique source reviews: 339

Candidate counts:
aspect
BUILDING       121
CHECKIN         65
CLEANLINESS     49
FOOD           272
LOCATION       247
ROOMS          448
SERVICE        374
VALUE          117
Name: count, dtype: int64


### 10.1 Balanced Construction Target

The smallest retained category contains 49 eligible segments. Rather than using all available segments and allowing common categories such as ROOMS and SERVICE to dominate, the controlled experiment targets approximately equal representation across the eight selected aspects.

The initial construction target is 100 synthetic multi-aspect documents containing three distinct aspect segments each. This produces 300 aspect occurrences, corresponding to approximately 37 or 38 examples per aspect. This target remains below the available candidate count for every retained category while providing enough constructed examples to demonstrate systematic dataset generation.

In [21]:
from collections import defaultdict

rng = np.random.default_rng(RANDOM_SEED)

# Shuffle candidates independently within each aspect
aspect_pools = {}

for aspect in construction_aspects:
    pool = (
        construction_candidates[
            construction_candidates["aspect"] == aspect
        ]
        .sample(frac=1, random_state=RANDOM_SEED)
        .reset_index(drop=True)
    )

    aspect_pools[aspect] = pool


# Create 100 three-aspect combinations.
# The offsets ensure that each constructed document contains
# three different aspects while keeping representation balanced.
aspect_sequences = []

for i in range(100):
    a1 = construction_aspects[i % 8]
    a2 = construction_aspects[(i + 3) % 8]
    a3 = construction_aspects[(i + 5) % 8]

    aspect_sequences.append([a1, a2, a3])


# Track the next unused row for each aspect
pool_positions = defaultdict(int)

constructed_rows = []

for doc_index, aspect_sequence in enumerate(aspect_sequences):

    selected_segments = []
    used_reviews = set()

    for order, aspect in enumerate(aspect_sequence, start=1):

        pool = aspect_pools[aspect]
        start_position = pool_positions[aspect]

        selected_row = None

        # Find the next unused segment from a different source review
        for pos in range(start_position, len(pool)):
            candidate = pool.iloc[pos]

            if candidate["review_id"] not in used_reviews:
                selected_row = candidate
                pool_positions[aspect] = pos + 1
                break

        if selected_row is None:
            raise ValueError(
                f"No suitable unused segment found for {aspect}"
            )

        used_reviews.add(selected_row["review_id"])

        selected_segments.append({
            "constructed_id": f"C{doc_index + 1:03d}",
            "segment_order": order,
            "aspect": aspect,
            "source_review_id": selected_row["review_id"],
            "source_segment_index": selected_row["segment_index"],
            "segment_text": selected_row["segment_text"],
            "segment_token_count": selected_row["token_count"]
        })

    constructed_rows.extend(selected_segments)


constructed_segments_df = pd.DataFrame(constructed_rows)

print("Constructed documents:",
      constructed_segments_df["constructed_id"].nunique())

print("Constructed segments:",
      len(constructed_segments_df))

print("\nAspect occurrences:")
print(
    constructed_segments_df["aspect"]
    .value_counts()
    .reindex(construction_aspects)
)

Constructed documents: 100
Constructed segments: 300

Aspect occurrences:
aspect
BUILDING       38
CHECKIN        37
CLEANLINESS    37
FOOD           38
LOCATION       37
ROOMS          38
SERVICE        38
VALUE          37
Name: count, dtype: int64


### 10.2 Construction Constraints

Each constructed document contains exactly three segments representing three distinct hotel aspects. Segments within the same constructed document are also drawn from different source reviews. This prevents a synthetic document from unintentionally preserving contextual continuity from a single original review.

The resulting documents therefore contain controlled and known aspect transitions while retaining naturally written TripAdvisor sentences.

The original source review identifier and source segment index are preserved for every constructed segment, providing traceability back to the original annotated corpus.

## 11. Construction of Complete Multi-Aspect Documents

The selected segments are now concatenated to form complete synthetic multi-aspect documents. Each constructed document contains three segments representing three distinct hotel aspects.

Because the identity and order of the source segments are known, the exact transition points between aspects can be retained as ground-truth boundaries. These boundaries provide a controlled reference for later analysis of whether C-Top2Vec changes its topic assignments near genuine aspect transitions.

Both token-level and character-level boundary positions are recorded. Token boundaries are based on the whitespace-delimited representation used during the exploratory length analysis.

In [22]:
constructed_documents = []

for constructed_id, group in constructed_segments_df.groupby(
    "constructed_id",
    sort=True
):
    group = group.sort_values("segment_order")

    texts = group["segment_text"].tolist()
    aspects = group["aspect"].tolist()
    source_reviews = group["source_review_id"].tolist()

    token_lengths = group["segment_token_count"].tolist()

    # Complete document
    constructed_text = " ".join(texts)

    # Token boundaries:
    # boundary_1 = first token position belonging to aspect 2
    # boundary_2 = first token position belonging to aspect 3
    boundary_token_1 = token_lengths[0]
    boundary_token_2 = token_lengths[0] + token_lengths[1]

    # Character boundaries in the concatenated string
    boundary_char_1 = len(texts[0])
    boundary_char_2 = len(texts[0]) + 1 + len(texts[1])

    constructed_documents.append({
        "constructed_id": constructed_id,
        "constructed_text": constructed_text,

        "aspect_1": aspects[0],
        "aspect_2": aspects[1],
        "aspect_3": aspects[2],

        "aspect_sequence": " -> ".join(aspects),

        "source_review_1": source_reviews[0],
        "source_review_2": source_reviews[1],
        "source_review_3": source_reviews[2],

        "segment_1_tokens": token_lengths[0],
        "segment_2_tokens": token_lengths[1],
        "segment_3_tokens": token_lengths[2],

        "boundary_token_1": boundary_token_1,
        "boundary_token_2": boundary_token_2,

        "boundary_char_1": boundary_char_1,
        "boundary_char_2": boundary_char_2,

        "total_tokens": sum(token_lengths)
    })

constructed_documents_df = pd.DataFrame(constructed_documents)

print("Constructed documents:",
      len(constructed_documents_df))

print(
    "Mean document length:",
    round(constructed_documents_df["total_tokens"].mean(), 2)
)

print(
    "Median document length:",
    constructed_documents_df["total_tokens"].median()
)

print(
    "Minimum document length:",
    constructed_documents_df["total_tokens"].min()
)

print(
    "Maximum document length:",
    constructed_documents_df["total_tokens"].max()
)

constructed_documents_df.head()

Constructed documents: 100
Mean document length: 40.99
Median document length: 40.5
Minimum document length: 17
Maximum document length: 64


,constructed_id,constructed_text,aspect_1,aspect_2,aspect_3,aspect_sequence,source_review_1,source_review_2,source_review_3,segment_1_tokens,segment_2_tokens,segment_3_tokens,boundary_token_1,boundary_token_2,boundary_char_1,boundary_char_2,total_tokens
0,C001,The lobby was a little small and got crowded o...,BUILDING,FOOD,ROOMS,BUILDING -> FOOD -> ROOMS,94326:2,582505:15,618542:152,17,14,9,17,31,95,181,40
1,C002,"Our check in was done in the room, we never ha...",CHECKIN,LOCATION,SERVICE,CHECKIN -> LOCATION -> SERVICE,81126:41,275848:198,113940:24,17,17,11,17,34,74,174,45
2,C003,I think that towels were just refolded and rep...,CLEANLINESS,ROOMS,VALUE,CLEANLINESS -> ROOMS -> VALUE,111418:22,247957:133,542696:0,14,18,7,14,32,81,182,39
3,C004,"They cook up beef, chicken, seafood, cut up ve...",FOOD,SERVICE,BUILDING,FOOD -> SERVICE -> BUILDING,218486:79,275848:198,81444:144,12,17,5,12,29,67,160,34
4,C005,"Great location - plenty to see, and good train...",LOCATION,VALUE,CHECKIN,LOCATION -> VALUE -> CHECKIN,192044:88,100581:9,112325:48,10,24,13,10,34,59,193,47


### 11.1 Validation of Constructed Aspect Boundaries

The stored token boundaries are validated against the original segment lengths. For a three-segment document, the first boundary should occur immediately after the tokens belonging to the first aspect, while the second boundary should occur immediately after the combined tokens belonging to the first and second aspects.

The constructed documents are also checked to confirm that each contains exactly three distinct aspects and three distinct source reviews.

In [23]:
validation = constructed_documents_df.copy()

validation["boundary_1_valid"] = (
    validation["boundary_token_1"]
    == validation["segment_1_tokens"]
)

validation["boundary_2_valid"] = (
    validation["boundary_token_2"]
    == (
        validation["segment_1_tokens"]
        + validation["segment_2_tokens"]
    )
)

validation["three_distinct_aspects"] = validation.apply(
    lambda row: len({
        row["aspect_1"],
        row["aspect_2"],
        row["aspect_3"]
    }) == 3,
    axis=1
)

validation["three_distinct_reviews"] = validation.apply(
    lambda row: len({
        row["source_review_1"],
        row["source_review_2"],
        row["source_review_3"]
    }) == 3,
    axis=1
)

print(
    "Valid first boundaries:",
    validation["boundary_1_valid"].sum(),
    "/",
    len(validation)
)

print(
    "Valid second boundaries:",
    validation["boundary_2_valid"].sum(),
    "/",
    len(validation)
)

print(
    "Documents with three distinct aspects:",
    validation["three_distinct_aspects"].sum(),
    "/",
    len(validation)
)

print(
    "Documents from three distinct source reviews:",
    validation["three_distinct_reviews"].sum(),
    "/",
    len(validation)
)

Valid first boundaries: 100 / 100
Valid second boundaries: 100 / 100
Documents with three distinct aspects: 100 / 100
Documents from three distinct source reviews: 100 / 100


### 11.2 Example Constructed Documents

A small sample of constructed documents is inspected manually to confirm that aspect transitions remain interpretable after concatenation. The known aspect sequence is displayed alongside the complete synthetic document.

In [24]:
for _, row in constructed_documents_df.head(5).iterrows():

    print("=" * 90)
    print("Constructed ID:", row["constructed_id"])
    print("Aspect sequence:", row["aspect_sequence"])

    print(
        "Token boundaries:",
        row["boundary_token_1"],
        "and",
        row["boundary_token_2"]
    )

    print("\nText:")
    print(row["constructed_text"])
    print()

Constructed ID: C001
Aspect sequence: BUILDING -> FOOD -> ROOMS
Token boundaries: 17 and 31

Text:
The lobby was a little small and got crowded one monrning whena larger group of people arrived. Breakfast buffet in hotel restaurant had everything you would want to start your day. We stayed in room 408 and it was great.

Constructed ID: C002
Aspect sequence: CHECKIN -> LOCATION -> SERVICE
Token boundaries: 17 and 34

Text:
Our check in was done in the room, we never had to stop by the front desk. The location is best, near the train station (5 minutes walk) and many budget eating houses nearby. From the moment we booked the room we were treated wonderfully.

Constructed ID: C003
Aspect sequence: CLEANLINESS -> ROOMS -> VALUE
Token boundaries: 14 and 32

Text:
I think that towels were just refolded and replaced instead of giving new towels. If you expect 5 star service then don't leave your country where you have all the modern facilities. Overall, the room was worth the price.

Construc

### 11.3 Constructed Document Length Relative to the Contextual Window

The constructed documents have a mean length of approximately 41 whitespace-delimited tokens. Because the original C-Top2Vec configuration uses a contextual window of 50 tokens, the constructed-document lengths are compared against this threshold.

Documents shorter than or equal to 50 tokens allow all three controlled aspect segments to occur within approximately one contextual window. Longer documents remain useful because their known aspect boundaries allow topic-transition behaviour to be examined across adjacent contextual regions.

Whitespace-token counts are used here as an approximate descriptive measure and should not be interpreted as identical to the subword tokenisation used internally by the embedding model.

In [25]:
within_50 = (constructed_documents_df["total_tokens"] <= 50).sum()
over_50 = (constructed_documents_df["total_tokens"] > 50).sum()

print("Documents <= 50 tokens:", within_50)
print("Documents > 50 tokens:", over_50)

print(
    "Percentage <= 50 tokens:",
    round(within_50 / len(constructed_documents_df) * 100, 2),
    "%"
)

print(
    "Percentage > 50 tokens:",
    round(over_50 / len(constructed_documents_df) * 100, 2),
    "%"
)

Documents <= 50 tokens: 85
Documents > 50 tokens: 15
Percentage <= 50 tokens: 85.0 %
Percentage > 50 tokens: 15.0 %


## 12. Saving the Processed and Constructed Datasets

To make the new-data pipeline reproducible, the processed sentence-level TripAdvisor corpus and both representations of the controlled constructed dataset are saved locally.

Three outputs are retained:

1. the processed sentence-level TripAdvisor data,
2. the segment-level representation of the controlled constructed documents, and
3. the complete document-level representation containing the known aspect sequence and boundary positions.

Retaining both segment-level and document-level representations preserves traceability from each synthetic document back to its original TripAdvisor source segments.

In [26]:
from pathlib import Path

processed_dir = (
    Path.home()
    / "Documents"
    / "contextual-top2vec-reproduction"
    / "data"
    / "processed"
)

processed_dir.mkdir(parents=True, exist_ok=True)

segments_path = processed_dir / "tripadvisor_segments_processed.csv"
constructed_segments_path = processed_dir / "constructed_segments.csv"
constructed_documents_path = processed_dir / "constructed_documents.csv"

segments_df.to_csv(
    segments_path,
    index=False
)

constructed_segments_df.to_csv(
    constructed_segments_path,
    index=False
)

constructed_documents_df.to_csv(
    constructed_documents_path,
    index=False
)

print("Saved:")
print(segments_path)
print(constructed_segments_path)
print(constructed_documents_path)

Saved:
/Users/cv/Documents/contextual-top2vec-reproduction/data/processed/tripadvisor_segments_processed.csv
/Users/cv/Documents/contextual-top2vec-reproduction/data/processed/constructed_segments.csv
/Users/cv/Documents/contextual-top2vec-reproduction/data/processed/constructed_documents.csv


### 12.1 Saved Dataset Verification

The saved files are reloaded to confirm that the processed datasets can be recovered independently of the notebook state. This provides a reproducible checkpoint before model experimentation begins.

In [27]:
saved_segments = pd.read_csv(segments_path)
saved_constructed_segments = pd.read_csv(constructed_segments_path)
saved_constructed_documents = pd.read_csv(constructed_documents_path)

print("Processed TripAdvisor segments:",
      saved_segments.shape)

print("Constructed segment dataset:",
      saved_constructed_segments.shape)

print("Constructed document dataset:",
      saved_constructed_documents.shape)

print("\nConstructed document IDs unique:",
      saved_constructed_documents["constructed_id"].is_unique)

print(
    "Missing constructed text:",
    saved_constructed_documents["constructed_text"].isna().sum()
)

Processed TripAdvisor segments: (4856, 12)
Constructed segment dataset: (300, 7)
Constructed document dataset: (100, 17)

Constructed document IDs unique: True
Missing constructed text: 0


### 12.2 Constructed Dataset Summary

The controlled construction procedure produced 100 synthetic multi-aspect documents containing 300 annotated source segments. Each document contains three distinct hotel aspects drawn from three different source reviews, with exact aspect-transition boundaries retained for later evaluation.

The constructed documents have a mean length of 40.99 whitespace-delimited tokens, with lengths ranging from 17 to 64 tokens. Eighty-five percent of the documents contain 50 tokens or fewer, placing all three aspect segments within approximately one contextual window under the original C-Top2Vec window-size setting.

All 100 documents passed validation for both stored aspect boundaries, distinct aspect composition, and distinct source-review provenance. The resulting dataset therefore provides controlled ground truth for later analysis of topic changes around known semantic aspect transitions.

## 13. Reconstruction of Natural TripAdvisor Reviews

In addition to the controlled constructed dataset, the original TripAdvisor reviews are retained as a natural evaluation corpus. Sentence segments belonging to the same review are recombined in their original order to reconstruct complete review documents.

This natural corpus preserves the authentic progression of hotel-review content, including naturally occurring single-aspect and multi-aspect passages. It complements the controlled dataset, where aspect transitions are deliberately constructed and exactly known.

In [28]:
natural_reviews_df = (
    segments_df
    .sort_values(["review_id", "segment_index"])
    .groupby("review_id")
    .agg(
        split=("split", "first"),
        hotel_id=("hotel_id", "first"),
        annotator_id=("annotator_id", "first"),
        rating_overall=("rating_overall", "first"),
        review_text=("segment_text", lambda x: " ".join(x)),
        n_segments=("segment_text", "size"),
        total_tokens=("token_count", "sum")
    )
    .reset_index()
)

print("Natural TripAdvisor reviews:", len(natural_reviews_df))
print("Mean segments per review:",
      round(natural_reviews_df["n_segments"].mean(), 2))
print("Median segments per review:",
      natural_reviews_df["n_segments"].median())

print("Mean tokens per review:",
      round(natural_reviews_df["total_tokens"].mean(), 2))
print("Median tokens per review:",
      natural_reviews_df["total_tokens"].median())
print("Minimum tokens:",
      natural_reviews_df["total_tokens"].min())
print("Maximum tokens:",
      natural_reviews_df["total_tokens"].max())

natural_reviews_df.head()

Natural TripAdvisor reviews: 369
Mean segments per review: 13.16
Median segments per review: 10.0
Mean tokens per review: 202.91
Median tokens per review: 153.0
Minimum tokens: 17
Maximum tokens: 1365


,review_id,split,hotel_id,annotator_id,rating_overall,review_text,n_segments,total_tokens
0,100504:40,test,100504,1,5,"Lovely staff, great for business I stayed at t...",21,262
1,100505:94,train,100505,2,4,Great value in Seattle Myself and spouse staye...,17,317
2,100509:44,test,100509,1,1,"When we arrived, we climbed the flight of stai...",13,205
3,100560:20,train,100560,2,3,Good location - average rooms - Great Exec Lou...,8,152
4,100565:96,train,100565,2,5,great little hotel We were very pleased with t...,8,87


### 13.1 Review-Level Aspect Composition

The aspect annotations associated with each review are retained separately from the text. These labels are not supplied to C-Top2Vec during modelling. Instead, they provide reference information for later interpretation of whether discovered topics correspond to known hotel aspects.

In [29]:
review_aspects = (
    segments_df
    .groupby("review_id")["aspect_labels"]
    .apply(
        lambda lists: sorted(
            {
                aspect
                for labels in lists
                for aspect in labels
                if aspect not in {"OTHER", "NOTRELATED"}
            }
        )
    )
    .reset_index(name="annotated_aspects")
)

natural_reviews_df = natural_reviews_df.merge(
    review_aspects,
    on="review_id",
    how="left"
)

natural_reviews_df["n_annotated_aspects"] = (
    natural_reviews_df["annotated_aspects"].apply(len)
)

print(
    "Mean annotated aspects per review:",
    round(natural_reviews_df["n_annotated_aspects"].mean(), 2)
)

print(
    "Reviews containing multiple substantive aspects:",
    (natural_reviews_df["n_annotated_aspects"] > 1).sum()
)

natural_reviews_df[
    [
        "review_id",
        "total_tokens",
        "annotated_aspects",
        "n_annotated_aspects",
        "review_text"
    ]
].head(10)

Mean annotated aspects per review: 4.36
Reviews containing multiple substantive aspects: 355


,review_id,total_tokens,annotated_aspects,n_annotated_aspects,review_text
0,100504:40,262,"[BUSINESS, CLEANLINESS, ROOMS, SERVICE, VALUE]",5,"Lovely staff, great for business I stayed at t..."
1,100505:94,317,"[CHECKIN, FOOD, LOCATION, ROOMS, SERVICE, VALUE]",6,Great value in Seattle Myself and spouse staye...
2,100509:44,205,"[BUILDING, CHECKIN, ROOMS, SERVICE]",4,"When we arrived, we climbed the flight of stai..."
3,100560:20,152,"[BUILDING, BUSINESS, LOCATION, ROOMS, SERVICE,...",6,Good location - average rooms - Great Exec Lou...
4,100565:96,87,"[BUILDING, BUSINESS, FOOD, LOCATION, ROOMS, SE...",6,great little hotel We were very pleased with t...
5,100570:94,475,"[BUSINESS, CHECKIN, LOCATION, ROOMS, VALUE]",5,Great stay at the Westin! I was in town on bus...
6,100581:9,229,"[CHECKIN, CLEANLINESS, ROOMS, SERVICE, VALUE]",5,Kind and Considerate We booked this hotel afte...
7,100585:30,79,"[BUILDING, LOCATION, ROOMS, SERVICE]",4,Hotel Vintage Park I just got back from stayin...
8,100586:19,381,"[BUILDING, CHECKIN, FOOD, ROOMS, SERVICE]",5,great experience except......for the concierge...
9,100587:155,340,"[BUILDING, BUSINESS, CLEANLINESS, LOCATION, RO...",7,Unusual (and Very Good) Experience I stayed at...


In [30]:
natural_reviews_path = processed_dir / "tripadvisor_natural_reviews.csv"

natural_reviews_df.to_csv(
    natural_reviews_path,
    index=False
)

print("Saved:")
print(natural_reviews_path)
print("Shape:", natural_reviews_df.shape)

Saved:
/Users/cv/Documents/contextual-top2vec-reproduction/data/processed/tripadvisor_natural_reviews.csv
Shape: (369, 10)


## 14. New Dataset Preparation Summary

Two complementary datasets have now been prepared for the C-Top2Vec extension.

The natural TripAdvisor corpus contains 369 reconstructed hotel reviews derived from 4,856 annotated sentence segments. Reviews contain an average of 13.16 segments and 202.91 whitespace-delimited tokens. The annotation analysis indicates that the corpus is strongly multi-aspect: 355 of the 369 reviews contain more than one substantive hotel aspect, with an average of 4.36 annotated aspects per review.

A second controlled dataset was constructed from clearly annotated single-aspect segments. It contains 100 synthetic documents, each composed of three distinct hotel aspects drawn from three different source reviews. Eight substantive aspects are represented approximately equally, and the exact transition boundaries between aspects are preserved.

The controlled documents have a mean length of 40.99 tokens, and 85 of the 100 documents contain no more than 50 whitespace-delimited tokens. This provides a controlled complement to the longer natural reviews and allows later analysis of whether C-Top2Vec topic assignments change near known aspect boundaries.

Together, the datasets provide two forms of evaluation: naturally occurring multi-aspect reviews and controlled documents with explicitly known semantic transitions.

## 15. C-Top2Vec Extension to Natural TripAdvisor Reviews

The reconstructed natural TripAdvisor reviews are now used as an unseen domain extension for C-Top2Vec. Aspect annotations are not supplied to the model during training. They are retained only as reference information for subsequent interpretation.

For this initial extension run, the reduced feasibility configuration used in the earlier 20 Newsgroups experiment is retained. These settings differ from the full configuration reported in the original study and are intended to accommodate local computational constraints while maintaining consistency with the completed feasibility experiment.

### 15.1 Natural Review Preflight Check

In [31]:
tripadvisor_documents = natural_reviews_df["review_text"].tolist()
tripadvisor_ids = natural_reviews_df["review_id"].tolist()

print("Documents:", len(tripadvisor_documents))
print(
    "Approximate whitespace tokens:",
    natural_reviews_df["total_tokens"].sum()
)

print(
    "Mean tokens per document:",
    round(natural_reviews_df["total_tokens"].mean(), 2)
)

print(
    "Maximum tokens in one document:",
    natural_reviews_df["total_tokens"].max()
)

print("\nFirst document ID:", tripadvisor_ids[0])
print(tripadvisor_documents[0][:500])

Documents: 369
Approximate whitespace tokens: 74872
Mean tokens per document: 202.91
Maximum tokens in one document: 1365

First document ID: 100504:40
Lovely staff, great for business I stayed at the Hotel Monaco Seattle during a conference for work. I otherwise probably would not have been able to stay there for four nights (it isn't cheap), but with work paying for it, I got to splurge a little. When I arrived the first night, it was just in time for their nightly wine happy hour. The woman at the desk let me know there were 10 minutes left and that I might want to swing through there before going to my room to get a couple glasses to take u


### 15.2 Model Configuration

The extension uses the same reduced configuration as the successful 20 Newsgroups feasibility run:

- `all-mpnet-base-v2` contextual embeddings
- contextual Top2Vec enabled
- smoothing window of 3
- minimum word count of 5
- UMAP with 15 neighbours and 5 components
- HDBSCAN minimum cluster size of 5
- embedding batch size of 4

Using the same feasibility configuration allows the TripAdvisor extension to be interpreted consistently with the earlier pilot while reducing memory demand on the local machine.

In [ ]:
from top2vec import Top2Vec
import time

umap_args = {
    "n_neighbors": 15,
    "n_components": 5
}

hdbscan_args = {
    "min_cluster_size": 5
}

start_time = time.time()

tripadvisor_model = Top2Vec(
    documents=tripadvisor_documents,
    contextual_top2vec=True,
    c_top2vec_smoothing_window=3,
    embedding_model="all-mpnet-base-v2",
    embedding_batch_size=4,
    min_count=5,
    umap_args=umap_args,
    hdbscan_args=hdbscan_args,
    workers=4,
    verbose=True
)

elapsed_minutes = (time.time() - start_time) / 60

print("\nTraining completed.")
print("Runtime:", round(elapsed_minutes, 2), "minutes")
print("Topics discovered:", tripadvisor_model.get_num_topics())

#### Initial Execution Result

The MPS execution did not complete. The run stopped during vocabulary embedding with an `MPS backend out of memory` error. This indicates a local hardware-memory constraint rather than a problem with the TripAdvisor dataset or preprocessing pipeline.

A CPU fallback is therefore used in the following section while retaining the same dataset, embedding model, and C-Top2Vec configuration.

### 15.3 Computational Constraint and CPU Fallback

The initial TripAdvisor extension run could not complete on the Apple MPS backend because the contextual embedding stage exceeded the available GPU memory. The failure occurred during vocabulary embedding rather than during topic clustering.

To preserve the selected `all-mpnet-base-v2` embedding model and the established C-Top2Vec configuration, the extension is rerun using CPU computation. This changes only the hardware used for embedding and does not alter the input corpus, embedding model, contextual configuration, UMAP parameters, or HDBSCAN parameters.

This computational adjustment is recorded as part of the feasibility limitations of the local experimental environment.

In [33]:
from pathlib import Path
import pandas as pd
import torch
import time

# Project paths
project_dir = (
    Path.home()
    / "Documents"
    / "contextual-top2vec-reproduction"
)

processed_dir = project_dir / "data" / "processed"
model_dir = project_dir / "models"

model_dir.mkdir(parents=True, exist_ok=True)

# Reload saved natural TripAdvisor dataset
natural_reviews_path = (
    processed_dir / "tripadvisor_natural_reviews.csv"
)

natural_reviews_df = pd.read_csv(natural_reviews_path)

tripadvisor_documents = (
    natural_reviews_df["review_text"]
    .astype(str)
    .tolist()
)

tripadvisor_ids = (
    natural_reviews_df["review_id"]
    .astype(str)
    .tolist()
)

print("Natural reviews loaded:", len(tripadvisor_documents))
print("MPS originally available:", torch.backends.mps.is_available())

Natural reviews loaded: 369
MPS originally available: True


In [34]:
# Force Top2Vec's contextual embedding functions to use CPU.
# Top2Vec checks torch.backends.mps.is_available() when choosing its device.
torch.backends.mps.is_available = lambda: False

print("MPS available to Top2Vec after override:",
      torch.backends.mps.is_available())

from top2vec import Top2Vec

MPS available to Top2Vec after override: False


In [35]:
umap_args = {
    "n_neighbors": 15,
    "n_components": 5
}

hdbscan_args = {
    "min_cluster_size": 5
}

start_time = time.time()

tripadvisor_model = Top2Vec(
    documents=tripadvisor_documents,
    contextual_top2vec=True,
    c_top2vec_smoothing_window=3,
    embedding_model="all-mpnet-base-v2",
    embedding_batch_size=4,
    min_count=5,
    umap_args=umap_args,
    hdbscan_args=hdbscan_args,
    workers=4,
    verbose=True
)

elapsed_minutes = (time.time() - start_time) / 60

print("\nTraining completed successfully.")
print("Runtime:", round(elapsed_minutes, 2), "minutes")
print("Topics discovered:",
      tripadvisor_model.get_num_topics())

2026-10-01 21:28:31,005 - top2vec - INFO - Pre-processing documents for training
/usr/local/Caskroom/miniforge/base/envs/ctop2vec-env/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-10-01 21:28:31,179 - top2vec - INFO - Creating vocabulary embedding
Embedding vocabulary: 100%|████████████████████████████████████████████████████████████| 41/41 [11:22<00:00, 16.65s/it]
2026-10-01 21:39:55,686 - top2vec - INFO - Create contextualized document embeddings
Embedding documents: 100%|█████████████████████████████████████████████████████████████| 12/12 [03:07<00:00, 15.66s/it]
369it [00:00, 3709.10it/s]
2026-10-01 21:43:12,595 - top2vec - INFO - Creating lower dimension embedding of documents
2026-10-01 21:43:29,608 - top2vec - INFO - Finding dense areas of documents
2026-10-01 21:43:29,678 - top2vec - INFO - Finding topics
Calculating document topic distribution


Training completed successfully.
Runtime: 15.03 minutes
Topics discovered: 188


In [36]:
tripadvisor_model_path = (
    model_dir / "ctop2vec_tripadvisor_natural_369"
)

tripadvisor_model.save(str(tripadvisor_model_path))

print("Model saved successfully.")
print("Path:", tripadvisor_model_path)
print("Topics:", tripadvisor_model.get_num_topics())

Model saved successfully.
Path: /Users/cv/Documents/contextual-top2vec-reproduction/models/ctop2vec_tripadvisor_natural_369
Topics: 188


### 15.4 Natural TripAdvisor Extension Result

The CPU-based C-Top2Vec extension completed successfully on all 369 reconstructed TripAdvisor reviews. Training required approximately 15.03 minutes and discovered 188 topics.

The model therefore successfully transferred the contextual topic-modelling pipeline from the 20 Newsgroups feasibility experiment to a new consumer-review domain.

However, the number of discovered topics remains large relative to the 369 input documents and the smaller set of annotated hotel aspects. This suggests that the reduced feasibility configuration continues to produce a relatively fragmented topic space. The result is therefore treated as an exploratory extension rather than evidence that the discovered topics correspond one-to-one with the TripAdvisor aspect annotations.

## 16. Inspection of Discovered TripAdvisor Topics

The discovered topics are inspected qualitatively before comparison with the annotated TripAdvisor aspects. Topic words provide an initial indication of whether the model has identified coherent semantic themes such as rooms, service, food, location, cleanliness, or value.

Because the model discovered considerably more topics than the number of annotated hotel aspects, this inspection also helps identify possible fragmentation or overlapping topic representations.

In [38]:
topic_words, word_scores, topic_nums = (
    tripadvisor_model.get_topics(20)
)

print("Total topics discovered:", tripadvisor_model.get_num_topics())
print("Topics currently displayed:", len(topic_nums))

for i in range(min(15, len(topic_nums))):
    print("\nTopic", topic_nums[i])
    print(", ".join(topic_words[i][:10]))

Total topics discovered: 188
Topics currently displayed: 20

Topic 0
resorts, resort, waikiki, hotels, hotel, westin, marriott, hyatt, beaches, jacuzzi

Topic 1
hotels, tokyo, attractions, marriott, okura, amenities, hotel, hyatt, decor, westin

Topic 2
resorts, hotel, hotels, honeymoon, resort, marriott, hyatt, attractions, amenities, jacuzzi

Topic 3
resort, resorts, hotel, hotels, amenities, marriott, attractions, westin, hyatt, luxurious

Topic 4
marriott, amenities, hotel, luxurious, resort, attractions, westin, resorts, hotels, hyatt

Topic 5
resort, resorts, hotel, hotels, attractions, marriott, beaches, jacuzzi, honeymoon, bath

Topic 6
punta, dominican, resorts, hotels, marriott, resort, hotel, buffet, attractions, restaurants

Topic 7
resorts, resort, punta, caribbean, attractions, dominican, amenities, buffets, hotels, marriott

Topic 8
resorts, resort, attractions, tours, tour, punta, beaches, exploring, hotels, paradise

Topic 9
boston, hotel, hotels, marriott, westin, dow

### 16.1 Preliminary Qualitative Interpretation

Inspection of the largest discovered topics indicates substantial semantic overlap. Frequently occurring terms such as *hotel*, *resort*, *Marriott*, *Hyatt*, *Westin*, *attractions*, and *amenities* appear across multiple topics.

Several large topics also appear to capture destination or property context, including locations such as Waikiki, Boston, Tokyo, Amsterdam, and Punta Cana, rather than corresponding directly to the annotated TripAdvisor aspects such as ROOMS, SERVICE, FOOD, CLEANLINESS, or VALUE.

This suggests that the unsupervised topic structure is considerably more fine-grained and fragmented than the human-defined aspect taxonomy. The 188 discovered topics should therefore not be interpreted as direct equivalents of the annotated hotel aspects.

## 17. Relationship Between Discovered Topics and Annotated Aspects

The TripAdvisor aspect annotations were not provided to C-Top2Vec during training. They can therefore be used after modelling as an external reference for interpreting the discovered topic structure.

For each natural review, the dominant C-Top2Vec topic is identified from its document-topic distribution and relevance scores. The annotated hotel aspects associated with reviews assigned to each dominant topic are then aggregated.

Because individual reviews may contain several annotated aspects, this analysis is descriptive rather than a one-to-one classification evaluation.

### 17.1 Restore Review-Level Aspect Labels

The natural review dataset was reloaded from CSV before the CPU-based model run. Since list-valued columns are stored as text in CSV format, the annotated aspect lists are converted back into Python lists before comparison with the discovered topics.

In [42]:
import ast

natural_reviews_df["annotated_aspects"] = (
    natural_reviews_df["annotated_aspects"]
    .apply(
        lambda x: ast.literal_eval(x)
        if isinstance(x, str)
        else x
    )
)

print(
    "Example restored aspect list:",
    natural_reviews_df.loc[0, "annotated_aspects"]
)

print(
    "Data type:",
    type(natural_reviews_df.loc[0, "annotated_aspects"])
)

Example restored aspect list: ['BUSINESS', 'CLEANLINESS', 'ROOMS', 'SERVICE', 'VALUE']
Data type: <class 'list'>


### 17.2 Dominant Topic Assignment

C-Top2Vec provides both a topic-distribution value and a relevance value for each topic within each document. Following the same procedure used in the earlier feasibility analysis, these values are multiplied and the highest-scoring topic is selected as the dominant topic for each review.

In [43]:
import numpy as np

topic_distribution = (
    tripadvisor_model.get_document_topic_distribution()
)

topic_relevance = (
    tripadvisor_model.get_document_topic_relevance()
)

print(
    "Topic distribution shape:",
    topic_distribution.shape
)

print(
    "Topic relevance shape:",
    topic_relevance.shape
)

combined_scores = (
    topic_distribution * topic_relevance
)

dominant_topics = np.argmax(
    combined_scores,
    axis=1
)

natural_reviews_df["dominant_topic"] = dominant_topics

print(
    "Reviews assigned:",
    len(dominant_topics)
)

print(
    "Unique dominant topics used:",
    natural_reviews_df["dominant_topic"].nunique()
)

Topic distribution shape: (369, 188)
Topic relevance shape: (369, 188)
Reviews assigned: 369
Unique dominant topics used: 188


### 17.3 Aspect Composition of Discovered Topics

The dominant-topic assignments are now compared with the human-annotated TripAdvisor aspects. For each discovered topic, the substantive aspect labels appearing in its assigned reviews are counted.

Because each review may contain multiple annotated aspects, the resulting percentages describe how frequently each aspect occurs among reviews associated with a topic rather than representing mutually exclusive class probabilities.

In [44]:
from collections import Counter
import pandas as pd

topic_aspect_rows = []

for topic_num, group in natural_reviews_df.groupby("dominant_topic"):

    aspect_counter = Counter()

    for aspects in group["annotated_aspects"]:
        for aspect in aspects:
            aspect_counter[aspect] += 1

    total_reviews = len(group)

    for aspect, count in aspect_counter.items():
        topic_aspect_rows.append({
            "topic": topic_num,
            "aspect": aspect,
            "aspect_count": count,
            "reviews_in_topic": total_reviews,
            "aspect_review_percentage": round(
                count / total_reviews * 100,
                2
            )
        })

topic_aspect_df = pd.DataFrame(topic_aspect_rows)

topic_aspect_df.head(20)

,topic,aspect,aspect_count,reviews_in_topic,aspect_review_percentage
0,0,BUILDING,1,1,100.0
1,0,CHECKIN,1,1,100.0
2,0,CLEANLINESS,1,1,100.0
3,0,FOOD,1,1,100.0
4,0,LOCATION,1,1,100.0
5,0,ROOMS,1,1,100.0
6,0,SERVICE,1,1,100.0
7,0,VALUE,1,1,100.0
8,1,BUILDING,1,1,100.0
9,1,FOOD,1,1,100.0


### 17.4 Most Common Annotated Aspect per Topic

To provide a compact summary, the most frequently occurring human-annotated aspect is identified for each discovered topic.

In [45]:
top_aspect_per_topic = (
    topic_aspect_df
    .sort_values(
        ["topic", "aspect_review_percentage"],
        ascending=[True, False]
    )
    .groupby("topic")
    .first()
    .reset_index()
    .sort_values(
        ["reviews_in_topic", "aspect_review_percentage"],
        ascending=[False, False]
    )
)

top_aspect_per_topic.head(20)

,topic,aspect,aspect_count,reviews_in_topic,aspect_review_percentage
179,179,LOCATION,20,26,76.92
187,187,ROOMS,10,11,90.91
186,186,LOCATION,9,9,100.00
26,26,ROOMS,8,8,100.00
138,138,LOCATION,6,6,100.00
131,131,LOCATION,5,6,83.33
169,169,ROOMS,5,5,100.00
144,144,ROOMS,4,5,80.00
158,158,SERVICE,4,5,80.00
167,167,SERVICE,4,5,80.00


### 17.5 Interpretation of Topic-to-Aspect Correspondence

The topic-to-aspect comparison provides partial evidence that some discovered topics correspond to recognizable hotel-review aspects. For example, several of the larger dominant-topic groups are frequently associated with LOCATION, ROOMS, SERVICE, or FOOD annotations.

However, the model discovered 188 topics from only 369 reviews, and many topics are represented by very small numbers of documents. This confirms that the reduced feasibility configuration produces a highly fragmented topic space. High aspect percentages for very small topics should therefore be interpreted cautiously.

Overall, the TripAdvisor experiment demonstrates that the C-Top2Vec pipeline can be transferred successfully to a new multi-aspect consumer-review domain, while also revealing that further parameter tuning or topic consolidation would be required before treating the discovered topics as stable equivalents of the human-defined TripAdvisor aspects.

## 18. Controlled Multi-Aspect Boundary Evaluation

The controlled dataset contains 100 synthetic documents, each composed of three
segments representing three distinct TripAdvisor aspects. Because the exact
transition points between these aspects were retained during construction, the
dataset provides ground truth for evaluating whether C-Top2Vec changes its
token-level topic assignments near genuine aspect boundaries.

This experiment applies the same reduced C-Top2Vec configuration used for the
natural TripAdvisor extension. The resulting token-level topic assignments are
then compared with the two known aspect boundaries in each constructed document.

### 18.1 Load and Verify the Controlled Dataset

In [1]:
from pathlib import Path
import pandas as pd

project_dir = (
    Path.home()
    / "Documents"
    / "contextual-top2vec-reproduction"
)

processed_dir = project_dir / "data" / "processed"
model_dir = project_dir / "models"

constructed_path = (
    processed_dir / "constructed_documents.csv"
)

controlled_df = pd.read_csv(constructed_path)

controlled_documents = (
    controlled_df["constructed_text"]
    .astype(str)
    .tolist()
)

controlled_ids = (
    controlled_df["constructed_id"]
    .astype(str)
    .tolist()
)

print("Controlled documents loaded:", len(controlled_documents))
print("Unique IDs:", controlled_df["constructed_id"].nunique())

print(
    "Known aspect boundaries:",
    len(controlled_df) * 2
)

print(
    "Mean document length:",
    round(controlled_df["total_tokens"].mean(), 2)
)

print(
    "Documents <= 50 tokens:",
    (controlled_df["total_tokens"] <= 50).sum()
)

controlled_df[
    [
        "constructed_id",
        "aspect_sequence",
        "boundary_token_1",
        "boundary_token_2",
        "total_tokens"
    ]
].head()

Controlled documents loaded: 100
Unique IDs: 100
Known aspect boundaries: 200
Mean document length: 40.99
Documents <= 50 tokens: 85


,constructed_id,aspect_sequence,boundary_token_1,boundary_token_2,total_tokens
0,C001,BUILDING -> FOOD -> ROOMS,17,31,40
1,C002,CHECKIN -> LOCATION -> SERVICE,17,34,45
2,C003,CLEANLINESS -> ROOMS -> VALUE,14,32,39
3,C004,FOOD -> SERVICE -> BUILDING,12,29,34
4,C005,LOCATION -> VALUE -> CHECKIN,10,34,47


### 18.2 C-Top2Vec Model on Controlled Documents

In [2]:
import torch
import time

# Use CPU to avoid the Apple MPS memory issue encountered
# during the natural TripAdvisor experiment.
torch.backends.mps.is_available = lambda: False

from top2vec import Top2Vec

umap_args = {
    "n_neighbors": 15,
    "n_components": 5
}

hdbscan_args = {
    "min_cluster_size": 5
}

start_time = time.time()

controlled_model = Top2Vec(
    documents=controlled_documents,
    contextual_top2vec=True,
    c_top2vec_smoothing_window=3,
    embedding_model="all-mpnet-base-v2",
    embedding_batch_size=4,
    min_count=5,
    umap_args=umap_args,
    hdbscan_args=hdbscan_args,
    workers=4,
    verbose=True
)

controlled_runtime = (time.time() - start_time) / 60

print("\nTraining completed successfully.")
print("Runtime:", round(controlled_runtime, 2), "minutes")
print(
    "Topics discovered:",
    controlled_model.get_num_topics()
)

/usr/local/Caskroom/miniforge/base/envs/ctop2vec-env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-10-30 13:52:08,607 - top2vec - INFO - Pre-processing documents for training
/usr/local/Caskroom/miniforge/base/envs/ctop2vec-env/lib/python3.11/site-packages/sklearn/feature_extraction/text.py:527: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
2026-10-30 13:52:08,621 - top2vec - INFO - Creating vocabulary embedding
Embedding vocabulary: 100%|██████████████████████████████████████████████████████████████| 4/4 [01:01<00:00, 15.26s/it]
2026-10-30 13:53:11,337 - top2vec - INFO - Create contextualized document embeddings
Embedding documents: 100%|███████████████████████████████████████████████████████████████| 4/4 [00:51<00:00, 12.81s/it]
100i


Training completed successfully.
Runtime: 2.08 minutes
Topics discovered: 6


In [3]:
controlled_model_path = (
    model_dir / "ctop2vec_tripadvisor_controlled_100"
)

controlled_model.save(
    str(controlled_model_path)
)

print("Controlled model saved.")
print("Path:", controlled_model_path)
print(
    "Topics:",
    controlled_model.get_num_topics()
)

Controlled model saved.
Path: /Users/cv/Documents/contextual-top2vec-reproduction/models/ctop2vec_tripadvisor_controlled_100
Topics: 6


### 18.3 Inspect Topics Discovered in the Controlled Dataset

Before evaluating the known aspect boundaries, the six discovered topics are
inspected to determine whether their descriptive words correspond to the
TripAdvisor aspect categories represented in the controlled dataset.

In [4]:
num_controlled_topics = controlled_model.get_num_topics()

topic_words, topic_word_scores, topic_nums = (
    controlled_model.get_topics(num_controlled_topics)
)

print("Topics discovered:", num_controlled_topics)

for topic_num, words in zip(topic_nums, topic_words):
    print("\n" + "=" * 70)
    print("Topic", topic_num)
    print(", ".join(words[:15]))

Topics discovered: 6

Topic 0
hotel, parking, rooms, buffet, room, staff, bar, service, place, arrived, breakfast, floor, location, bathroom, around

Topic 1
hotel, rooms, buffet, room, parking, pool, bar, bathroom, breakfast, place, floor, food, location, lobby, staff

Topic 2
buffet, hotel, breakfast, rooms, room, food, staff, service, parking, bar, place, bathroom, desk, stayed, location

Topic 3
rooms, hotel, room, bathroom, floor, parking, buffet, clean, place, bar, desk, pool, lobby, staff, location

Topic 4
parking, hotel, rooms, floor, location, around, room, bar, place, in, buffet, arrived, price, bathroom, up

Topic 5
hotel, rooms, parking, room, price, buffet, location, floor, place, bar, around, stay, breakfast, lobby, night


### 18.4 Token-Level Output and Boundary Alignment

The controlled dataset stores ground-truth boundaries using whitespace-token
positions and exact character positions. C-Top2Vec, however, operates on
MPNet transformer tokens, which can split words into multiple subword tokens.

The model tokenisation is therefore aligned back to the original document text
before boundary-detection performance is calculated.

In [5]:
token_assignments = (
    controlled_model.get_document_token_topic_assignment()
)

model_tokens = controlled_model.get_document_tokens()

print("Documents with token assignments:",
      len(token_assignments))

print("Documents with token sequences:",
      len(model_tokens))

print("\nExample: C001")
print("Ground-truth aspect sequence:",
      controlled_df.loc[0, "aspect_sequence"])

print("Whitespace-token boundaries:",
      controlled_df.loc[0, "boundary_token_1"],
      controlled_df.loc[0, "boundary_token_2"])

print("Character boundaries:",
      controlled_df.loc[0, "boundary_char_1"],
      controlled_df.loc[0, "boundary_char_2"])

print("\nNumber of whitespace tokens:",
      controlled_df.loc[0, "total_tokens"])

print("Number of C-Top2Vec / MPNet tokens:",
      len(model_tokens[0]))

print("\nFirst 50 model tokens:")
print(model_tokens[0][:50])

print("\nToken-topic assignment structure:")
print(token_assignments[0])

Documents with token assignments: 100
Documents with token sequences: 100

Example: C001
Ground-truth aspect sequence: BUILDING -> FOOD -> ROOMS
Whitespace-token boundaries: 17 31
Character boundaries: 95 181

Number of whitespace tokens: 40
Number of C-Top2Vec / MPNet tokens: 49

First 50 model tokens:
['<s>', 'the', 'lobby', 'was', 'a', 'little', 'small', 'and', 'got', 'crowded', 'one', 'mon', '##rn', '##ing', 'when', '##a', 'larger', 'group', 'of', 'people', 'arrived', '.', 'breakfast', 'buffet', 'in', 'hotel', 'restaurant', 'had', 'everything', 'you', 'would', 'want', 'to', 'start', 'your', 'day', '.', 'we', 'stayed', 'in', 'room', '40', '##8', 'and', 'it', 'was', 'great', '.', '</s>']

Token-topic assignment structure:
Document(document_index=0, topics=[Topic(topic_index=2, tokens=array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
       17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33,
       34, 35, 36, 37, 38, 39, 44, 45, 46, 47, 48]), 

### 18.5 Align Transformer Tokens to Original Character Positions

C-Top2Vec token assignments are expressed using MPNet subword-token
positions, whereas the controlled dataset retains ground-truth boundaries in
the original document text. To compare them directly, the MPNet tokenizer is
rerun with character-offset information.

The resulting token sequence is checked against the token sequence returned by
C-Top2Vec before boundary metrics are calculated.

In [6]:
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained(
    "sentence-transformers/all-mpnet-base-v2",
    use_fast=True
)

aligned_documents = []
alignment_errors = []

for doc_idx, text in enumerate(controlled_documents):

    encoded = tokenizer(
        text,
        return_offsets_mapping=True,
        add_special_tokens=True,
        truncation=False
    )

    hf_tokens = tokenizer.convert_ids_to_tokens(
        encoded["input_ids"]
    )

    offsets = encoded["offset_mapping"]

    ctop_tokens = list(model_tokens[doc_idx])

    # Check whether the tokenizer output corresponds to
    # the tokens stored by C-Top2Vec.
    if len(hf_tokens) != len(ctop_tokens):
        alignment_errors.append(
            (
                doc_idx,
                "length",
                len(hf_tokens),
                len(ctop_tokens)
            )
        )

    else:
        mismatches = [
            i
            for i, (a, b) in enumerate(
                zip(hf_tokens, ctop_tokens)
            )
            if a != b
        ]

        if mismatches:
            alignment_errors.append(
                (
                    doc_idx,
                    "tokens",
                    mismatches[:10]
                )
            )

    aligned_documents.append(
        {
            "tokens": hf_tokens,
            "offsets": offsets
        }
    )

print("Documents checked:", len(aligned_documents))
print("Alignment problems:", len(alignment_errors))

if alignment_errors:
    print("\nFirst alignment problems:")
    print(alignment_errors[:5])
else:
    print("All C-Top2Vec tokens align with MPNet offsets.")

Documents checked: 100
Alignment problems: 0
All C-Top2Vec tokens align with MPNet offsets.


### 18.6 Extract Predicted Topic Boundaries

A predicted boundary is defined as a position where the C-Top2Vec token-level
topic assignment changes between adjacent non-special tokens.

Each predicted transition is mapped back to the corresponding position in the
original document and then to its whitespace-token boundary. This allows direct
comparison with the two known aspect boundaries in every controlled document.

In [7]:
import re

def get_topic_labels(document_assignment, n_tokens):
    """
    Convert C-Top2Vec's topic-grouped representation into
    one topic label for every transformer token.
    """

    labels = np.full(n_tokens, -1, dtype=int)

    for topic in document_assignment.topics:
        labels[topic.tokens] = topic.topic_index

    return labels


def whitespace_boundary_index(text, char_position):
    """
    Convert a character boundary into the number of
    whitespace-delimited tokens occurring before it.
    """

    spans = [
        match.span()
        for match in re.finditer(r"\S+", text)
    ]

    return sum(
        end <= char_position
        for _, end in spans
    )


predicted_boundary_rows = []

for doc_idx, row in controlled_df.iterrows():

    text = row["constructed_text"]

    tokens = aligned_documents[doc_idx]["tokens"]
    offsets = aligned_documents[doc_idx]["offsets"]

    labels = get_topic_labels(
        token_assignments[doc_idx],
        len(tokens)
    )

    predicted = []

    # Ignore special tokens, whose offsets are normally (0, 0)
    valid_indices = [
        i for i, (start, end) in enumerate(offsets)
        if not (start == 0 and end == 0)
    ]

    for left_idx, right_idx in zip(
        valid_indices[:-1],
        valid_indices[1:]
    ):

        if labels[left_idx] != labels[right_idx]:

            # Character position between the two tokens
            char_boundary = offsets[right_idx][0]

            ws_boundary = whitespace_boundary_index(
                text,
                char_boundary
            )

            predicted.append(
                {
                    "left_topic": int(labels[left_idx]),
                    "right_topic": int(labels[right_idx]),
                    "char_boundary": int(char_boundary),
                    "token_boundary": int(ws_boundary)
                }
            )

    predicted_boundary_rows.append(predicted)


print("Example C001")
print("True boundaries:",
      [
          int(controlled_df.loc[0, "boundary_token_1"]),
          int(controlled_df.loc[0, "boundary_token_2"])
      ])

print("Predicted topic transitions:")
print(predicted_boundary_rows[0])

Example C001
True boundaries: [17, 31]
Predicted topic transitions:
[{'left_topic': 2, 'right_topic': 3, 'char_boundary': 195, 'token_boundary': 34}, {'left_topic': 3, 'right_topic': 2, 'char_boundary': 208, 'token_boundary': 37}]


### 18.7 Quantitative Boundary Detection Evaluation

Boundary detection is evaluated by comparing C-Top2Vec topic-transition
positions with the two known aspect boundaries in each controlled document.

Because multiple transformer-token transitions may map to the same whitespace
position, predicted boundaries are deduplicated before evaluation.

A one-to-one ordered matching procedure is used so that one predicted boundary
cannot receive credit for multiple ground-truth boundaries. Performance is
reported for exact matches and tolerance windows of ±1, ±3, and ±5 whitespace
tokens using precision, recall, and F1-score.

In [8]:
from functools import lru_cache
import pandas as pd
import numpy as np


def optimal_ordered_matches(true_bounds, pred_bounds, tolerance):
    """
    Find a one-to-one ordered matching between true and predicted
    boundaries.

    Objective:
      1. maximise number of matched boundaries
      2. among equally sized matchings, minimise total distance
    """

    true_bounds = tuple(sorted(true_bounds))
    pred_bounds = tuple(sorted(pred_bounds))

    @lru_cache(None)
    def solve(i, j):

        if i >= len(true_bounds) or j >= len(pred_bounds):
            return 0, 0, ()

        candidates = []

        # Skip current true boundary
        candidates.append(
            solve(i + 1, j)
        )

        # Skip current predicted boundary
        candidates.append(
            solve(i, j + 1)
        )

        distance = abs(
            true_bounds[i] - pred_bounds[j]
        )

        # Match if within tolerance
        if distance <= tolerance:

            count, total_distance, pairs = solve(
                i + 1,
                j + 1
            )

            candidates.append(
                (
                    count + 1,
                    total_distance + distance,
                    (
                        (
                            true_bounds[i],
                            pred_bounds[j],
                            distance
                        ),
                    ) + pairs
                )
            )

        # Maximise matches, then minimise distance
        best = max(
            candidates,
            key=lambda x: (
                x[0],
                -x[1]
            )
        )

        return best

    return solve(0, 0)


# --------------------------------------------------
# Prepare boundary data for each controlled document
# --------------------------------------------------

boundary_eval_rows = []

for doc_idx, row in controlled_df.iterrows():

    true_bounds = sorted(
        [
            int(row["boundary_token_1"]),
            int(row["boundary_token_2"])
        ]
    )

    # Deduplicate transformer transitions that map to
    # the same whitespace-token boundary
    pred_bounds = sorted(
        set(
            transition["token_boundary"]
            for transition
            in predicted_boundary_rows[doc_idx]
        )
    )

    if pred_bounds:

        nearest_distances = [
            min(
                abs(true_boundary - predicted)
                for predicted in pred_bounds
            )
            for true_boundary in true_bounds
        ]

    else:

        nearest_distances = [
            np.nan,
            np.nan
        ]

    boundary_eval_rows.append(
        {
            "constructed_id":
                row["constructed_id"],

            "aspect_sequence":
                row["aspect_sequence"],

            "true_boundary_1":
                true_bounds[0],

            "true_boundary_2":
                true_bounds[1],

            "predicted_boundaries":
                pred_bounds,

            "n_predicted":
                len(pred_bounds),

            "nearest_distance_1":
                nearest_distances[0],

            "nearest_distance_2":
                nearest_distances[1]
        }
    )


boundary_eval_df = pd.DataFrame(
    boundary_eval_rows
)

print(
    "Documents evaluated:",
    len(boundary_eval_df)
)

print(
    "Ground-truth boundaries:",
    len(boundary_eval_df) * 2
)

print(
    "Total unique predicted boundaries:",
    boundary_eval_df["n_predicted"].sum()
)

print(
    "Mean predicted boundaries per document:",
    round(
        boundary_eval_df["n_predicted"].mean(),
        2
    )
)

print(
    "Documents with no predicted boundary:",
    (boundary_eval_df["n_predicted"] == 0).sum()
)

Documents evaluated: 100
Ground-truth boundaries: 200
Total unique predicted boundaries: 103
Mean predicted boundaries per document: 1.03
Documents with no predicted boundary: 72


In [10]:
# --------------------------------------------------
# Precision / Recall / F1 at different tolerances
# --------------------------------------------------

tolerances = {
    "Exact": 0,
    "±1 token": 1,
    "±3 tokens": 3,
    "±5 tokens": 5
}

metric_rows = []

total_true = len(boundary_eval_df) * 2

total_predicted = int(
    boundary_eval_df["n_predicted"].sum()
)

for label, tolerance in tolerances.items():

    total_matches = 0
    documents_both_matched = 0
    matched_distances = []

    for _, row in boundary_eval_df.iterrows():

        true_bounds = [
            int(row["true_boundary_1"]),
            int(row["true_boundary_2"])
        ]

        pred_bounds = row[
            "predicted_boundaries"
        ]

        (
            match_count,
            total_distance,
            pairs
        ) = optimal_ordered_matches(
            true_bounds,
            pred_bounds,
            tolerance
        )

        total_matches += match_count

        matched_distances.extend(
            [
                pair[2]
                for pair in pairs
            ]
        )

        if match_count == 2:
            documents_both_matched += 1

    precision = (
        total_matches / total_predicted
        if total_predicted > 0
        else 0
    )

    recall = (
        total_matches / total_true
        if total_true > 0
        else 0
    )

    f1 = (
        2 * precision * recall
        / (precision + recall)
        if precision + recall > 0
        else 0
    )

    metric_rows.append(
        {
            "Tolerance":
                label,

            "Matched boundaries":
                total_matches,

            "Precision":
                precision,

            "Recall":
                recall,

            "F1":
                f1,

            "Documents with both boundaries matched":
                documents_both_matched
        }
    )


boundary_metrics_df = pd.DataFrame(
    metric_rows
)

boundary_metrics_df[
    [
        "Precision",
        "Recall",
        "F1"
    ]
] = boundary_metrics_df[
    [
        "Precision",
        "Recall",
        "F1"
    ]
].round(3)

boundary_metrics_df

,Tolerance,Matched boundaries,Precision,Recall,F1,Documents with both boundaries matched
0,Exact,6,0.058,0.03,0.040,1
1,±1 token,18,0.175,0.09,0.119,5
2,±3 tokens,28,0.272,0.14,0.185,9
3,±5 tokens,36,0.350,0.18,0.238,11


In [9]:
all_nearest_distances = pd.concat(
    [
        boundary_eval_df["nearest_distance_1"],
        boundary_eval_df["nearest_distance_2"]
    ],
    ignore_index=True
).dropna()

print(
    "Mean nearest predicted-boundary distance:",
    round(all_nearest_distances.mean(), 2),
    "tokens"
)

print(
    "Median nearest predicted-boundary distance:",
    round(all_nearest_distances.median(), 2),
    "tokens"
)

print(
    "Maximum nearest predicted-boundary distance:",
    int(all_nearest_distances.max())
)

Mean nearest predicted-boundary distance: 5.59 tokens
Median nearest predicted-boundary distance: 3.5 tokens
Maximum nearest predicted-boundary distance: 24


### 18.8 Interpretation of Controlled Boundary Results

C-Top2Vec showed limited correspondence with the known aspect boundaries in
the controlled multi-aspect documents. Only 6 of the 200 ground-truth
boundaries were detected at their exact positions. Allowing a tolerance of
±5 whitespace tokens increased the number of matched boundaries to 36,
corresponding to a precision of 0.350, recall of 0.180, and F1-score of 0.238.

Seventy-two of the 100 controlled documents contained no predicted topic
transition despite each document containing three distinct annotated aspects.
Only 11 documents had both ground-truth boundaries matched within the
±5-token tolerance.

These results suggest that the latent topic transitions produced by C-Top2Vec
do not correspond closely to the human-defined TripAdvisor aspect boundaries
under the evaluated configuration. This complements the natural-review
experiment, where the model produced a highly fragmented topic space whose
topics frequently reflected hotel, destination, and amenity context rather
than the annotation scheme.

## 19. Topic Consolidation on Natural TripAdvisor Reviews

The natural TripAdvisor experiment produced 188 topics from 369 reviews,
indicating substantial topic fragmentation. As a follow-up analysis,
hierarchical topic reduction is applied to examine whether a smaller topic
space produces broader and more interpretable themes.

The model is reduced to 11 topics because the complete Trip-MA annotation
scheme contains 11 aspect categories. This target is therefore
ontology-informed and is used as a diagnostic comparison rather than as an
unsupervised estimate of the optimal number of topics.

### 19.1 Load the Saved Natural TripAdvisor Model

In [11]:
from top2vec import Top2Vec
from pathlib import Path

natural_model_path = (
    model_dir / "ctop2vec_tripadvisor_natural_369"
)

natural_reduced_model = Top2Vec.load(
    str(natural_model_path)
)

print(
    "Original natural-review topics:",
    natural_reduced_model.get_num_topics()
)

Original natural-review topics: 188


### 19.2 Hierarchical Topic Reduction

The 188 original topics are hierarchically consolidated to 11 reduced topics.
The original model is loaded from disk separately so that the previously saved
188-topic result remains unchanged.

In [12]:
target_reduced_topics = 11

hierarchy = (
    natural_reduced_model
    .hierarchical_topic_reduction(
        target_reduced_topics
    )
)

print(
    "Original topics:",
    natural_reduced_model.get_num_topics()
)

print(
    "Reduced topics:",
    natural_reduced_model.get_num_topics(
        reduced=True
    )
)

print(
    "Hierarchy groups:",
    len(hierarchy)
)

Calculating document topic distributions: 100%|████████████████████████████████████| 369/369 [00:00<00:00, 3616.73it/s]

Original topics: 188
Reduced topics: 11
Hierarchy groups: 11


In [18]:
reduced_words, reduced_scores, reduced_nums = (
    natural_reduced_model.get_topics(
        num_topics=11,
        reduced=True
    )
)

print("Reduced topics:", len(reduced_nums))

for topic_num, words in zip(
    reduced_nums,
    reduced_words
):
    print("\n" + "=" * 75)
    print(f"Reduced Topic {topic_num}")
    print(", ".join(words[:15]))

Reduced topics: 11

Reduced Topic 0
resorts, marriott, resort, hotels, hotel, attractions, punta, hyatt, westin, amenities, beaches, jacuzzi, spa, luxurious, suites

Reduced Topic 1
hotel, marriott, hotels, hyatt, westin, resort, resorts, attractions, amenities, chinatown, cafe, balcony, tokyo, luxurious, facilities

Reduced Topic 2
marriott, hotel, hyatt, hotels, westin, attractions, resorts, amenities, resort, palace, luxurious, facilities, cafe, rooms, inn

Reduced Topic 3
marriott, hotels, hotel, hyatt, resorts, resort, amenities, attractions, westin, reservations, facilities, complaints, guests, suites, rooms

Reduced Topic 4
marriott, hotel, hyatt, westin, hotels, balcony, amenities, attractions, york, resort, rooms, downtown, resorts, dining, inn

Reduced Topic 5
marriott, hotel, punta, hotels, hyatt, resort, madrid, westin, resorts, barcelona, attractions, ramblas, cafe, rambla, inn

Reduced Topic 6
resorts, resort, punta, attractions, beaches, marriott, hotels, hotel, spa, jac

In [14]:
hierarchy_rows = []

for reduced_topic, original_topics in enumerate(
    hierarchy
):

    hierarchy_rows.append(
        {
            "reduced_topic":
                reduced_topic,

            "n_original_topics_merged":
                len(original_topics),

            "original_topics":
                original_topics
        }
    )

hierarchy_df = pd.DataFrame(
    hierarchy_rows
)

hierarchy_df

,reduced_topic,n_original_topics_merged,original_topics
0,0,7,"[10, 51, 22, 49, 42, 154, 166]"
1,1,12,"[71, 73, 5, 4, 3, 1, 110, 72, 70, 60, 59, 171]"
2,2,8,"[130, 131, 170, 164, 175, 177, 176, 173]"
3,3,17,"[20, 19, 80, 15, 31, 55, 45, 18, 17, 2, 89, 91..."
4,4,14,"[9, 146, 145, 116, 117, 128, 118, 127, 113, 13..."
5,5,13,"[33, 106, 98, 46, 68, 105, 36, 57, 56, 54, 78,..."
6,6,21,"[7, 6, 48, 35, 52, 8, 29, 63, 34, 53, 43, 67, ..."
7,7,17,"[103, 136, 115, 121, 12, 162, 169, 40, 39, 21,..."
8,8,37,"[16, 0, 124, 92, 185, 160, 27, 81, 167, 144, 1..."
9,9,22,"[23, 24, 153, 142, 155, 150, 156, 13, 25, 69, ..."


In [19]:
reduced_topic_rows = []

for topic_num, words in zip(
    reduced_nums,
    reduced_words
):
    original_topics = hierarchy[
        int(topic_num)
    ]

    reduced_topic_rows.append(
        {
            "reduced_topic":
                int(topic_num),

            "n_original_topics_merged":
                len(original_topics),

            "top_15_words":
                ", ".join(words[:15])
        }
    )

reduced_topics_df = pd.DataFrame(
    reduced_topic_rows
)

reduced_topics_df

,reduced_topic,n_original_topics_merged,top_15_words
0,0,7,"resorts, marriott, resort, hotels, hotel, attr..."
1,1,12,"hotel, marriott, hotels, hyatt, westin, resort..."
2,2,8,"marriott, hotel, hyatt, hotels, westin, attrac..."
3,3,17,"marriott, hotels, hotel, hyatt, resorts, resor..."
4,4,14,"marriott, hotel, hyatt, westin, hotels, balcon..."
5,5,13,"marriott, hotel, punta, hotels, hyatt, resort,..."
6,6,21,"resorts, resort, punta, attractions, beaches, ..."
7,7,17,"marriott, hyatt, westin, hotel, hotels, attrac..."
8,8,37,"marriott, hotel, hyatt, westin, hotels, resort..."
9,9,22,"marriott, hotel, hotels, hyatt, westin, resort..."


In [20]:
reduced_topics_path = (
    processed_dir
    / "tripadvisor_reduced_11_topics.csv"
)

reduced_topics_df.to_csv(
    reduced_topics_path,
    index=False
)

print(
    "Saved:",
    reduced_topics_path
)

Saved: /Users/cv/Documents/contextual-top2vec-reproduction/data/processed/tripadvisor_reduced_11_topics.csv


In [21]:
reduced_model_path = (
    model_dir
    / "ctop2vec_tripadvisor_natural_369_reduced_11"
)

natural_reduced_model.save(
    str(reduced_model_path)
)

print(
    "Reduced model saved:",
    reduced_model_path
)

Reduced model saved: /Users/cv/Documents/contextual-top2vec-reproduction/models/ctop2vec_tripadvisor_natural_369_reduced_11


### 19.6 Interpretation of Topic Consolidation

Hierarchical reduction successfully consolidated the 188 natural-review topics
into 11 broader topics. However, the reduced topics did not become clearly
aligned with the 11 Trip-MA aspect categories.

The highest-ranked words remained dominated by hotel brands, resort and
destination terms, attractions, amenities, and general property vocabulary.
Distinct aspects such as Cleanliness, Value, Check-in, Business, and Food did
not emerge as clearly separated reduced topics.

Therefore, reducing the number of topics to match the size of the Trip-MA
annotation scheme reduced fragmentation but did not by itself resolve the
semantic mismatch between C-Top2Vec's latent topic structure and the
human-defined aspect taxonomy. Because the target of 11 topics was selected
using knowledge of the Trip-MA ontology, this experiment is treated as an
ontology-informed diagnostic rather than an unsupervised estimate of the
optimal topic count.

## 20. LLM-as-Judge Evaluation

An LLM-as-Judge evaluation is used as a complementary semantic assessment of
the 11 hierarchically reduced C-Top2Vec topics.

The boundary-detection experiment in Section 18 is evaluated directly against
known ground-truth positions and therefore does not require an LLM judge.
Instead, the LLM is used for the more subjective task of assessing whether the
words describing each reduced topic correspond meaningfully to one of the
human-defined Trip-MA aspect categories.

The judge is shown the top 15 words for one topic at a time and asked to select
the best-matching Trip-MA aspect and assign an alignment score:

- 0 = no meaningful alignment
- 1 = weak or mixed alignment
- 2 = moderate alignment
- 3 = strong, clearly aspect-specific alignment

The judge is not required to use every aspect and multiple topics may receive
the same aspect label.

### 20.1 Judge Prompt

The following fixed prompt is applied independently to each reduced topic:

"You are evaluating topics produced by an unsupervised topic model on
TripAdvisor hotel reviews.

Given the top 15 words of one topic, select the single Trip-MA aspect that best
describes the topic from:

ROOMS, CLEANLINESS, VALUE, SERVICE, LOCATION, CHECKIN, BUSINESS, FOOD,
BUILDING, OTHER, NOTRELATED.

Use the following alignment scale:

0 = no meaningful correspondence with the selected aspect
1 = weak or mixed correspondence
2 = moderate correspondence, although substantial ambiguity remains
3 = strong correspondence, with the topic clearly representing the aspect

Do not force a strong match. Hotel brands, destinations, and generic hotel
vocabulary alone are not sufficient evidence of a specific aspect.

Return:
best_aspect
alignment_score
confidence (1-5)
brief_reason"

In [22]:
llm_judge_results = [
    [0, "BUILDING", 1, 3,
     "Amenities, spa, jacuzzi and suites appear, but resorts, destinations and brands dominate."],

    [1, "LOCATION", 1, 2,
     "Attractions, Chinatown and Tokyo suggest location, but the topic remains highly mixed."],

    [2, "BUILDING", 1, 2,
     "Amenities and facilities appear, but brands and generic hotel terms dominate."],

    [3, "SERVICE", 1, 2,
     "Reservations, complaints and guests suggest service, but most words are generic property terms."],

    [4, "LOCATION", 1, 3,
     "Attractions, York and downtown provide location evidence, mixed with rooms and property terms."],

    [5, "LOCATION", 2, 4,
     "Punta, Madrid, Barcelona and Ramblas provide relatively coherent destination and location evidence."],

    [6, "LOCATION", 2, 5,
     "Punta, beaches, Caribbean, Dominican and tours provide the clearest geographic theme."],

    [7, "BUILDING", 1, 3,
     "Amenities, facilities, balcony and suites indicate property features, but the topic remains mixed."],

    [8, "BUILDING", 1, 3,
     "Amenities, facilities, suites and balcony appear, but hotel brands dominate."],

    [9, "LOCATION", 1, 3,
     "Berlin, Paris and attractions suggest location, but the evidence is not dominant."],

    [10, "BUILDING", 1, 3,
     "Amenities, facilities, balcony and suites suggest building or property features, mixed with brands and location terms."]
]

llm_judge_df = pd.DataFrame(
    llm_judge_results,
    columns=[
        "reduced_topic",
        "best_aspect",
        "alignment_score",
        "confidence",
        "brief_reason"
    ]
)

llm_judge_df

,reduced_topic,best_aspect,alignment_score,confidence,brief_reason
0,0,BUILDING,1,3,"Amenities, spa, jacuzzi and suites appear, but..."
1,1,LOCATION,1,2,"Attractions, Chinatown and Tokyo suggest locat..."
2,2,BUILDING,1,2,"Amenities and facilities appear, but brands an..."
3,3,SERVICE,1,2,"Reservations, complaints and guests suggest se..."
4,4,LOCATION,1,3,"Attractions, York and downtown provide locatio..."
5,5,LOCATION,2,4,"Punta, Madrid, Barcelona and Ramblas provide r..."
6,6,LOCATION,2,5,"Punta, beaches, Caribbean, Dominican and tours..."
7,7,BUILDING,1,3,"Amenities, facilities, balcony and suites indi..."
8,8,BUILDING,1,3,"Amenities, facilities, suites and balcony appe..."
9,9,LOCATION,1,3,"Berlin, Paris and attractions suggest location..."


In [23]:
print(
    "Mean alignment score:",
    round(
        llm_judge_df["alignment_score"].mean(),
        2
    ),
    "/ 3"
)

print("\nAlignment score counts:")
print(
    llm_judge_df["alignment_score"]
    .value_counts()
    .sort_index()
)

print("\nAssigned aspects:")
print(
    llm_judge_df["best_aspect"]
    .value_counts()
)

Mean alignment score: 1.18 / 3

Alignment score counts:
alignment_score
1    9
2    2
Name: count, dtype: int64

Assigned aspects:
best_aspect
BUILDING    5
LOCATION    5
SERVICE     1
Name: count, dtype: int64


In [24]:
llm_judge_path = (
    processed_dir
    / "tripadvisor_llm_judge_results.csv"
)

llm_judge_df.to_csv(
    llm_judge_path,
    index=False
)

print("Saved:", llm_judge_path)

Saved: /Users/cv/Documents/contextual-top2vec-reproduction/data/processed/tripadvisor_llm_judge_results.csv


### 20.2 Interpretation of LLM-as-Judge Results

The LLM-as-Judge evaluation provided limited evidence that the 11 reduced
C-Top2Vec topics correspond to the human-defined Trip-MA aspect taxonomy.

None of the 11 reduced topics received a strong alignment score of 3. Nine
topics were rated as weakly or ambiguously aligned, while only two received a
moderate score. The mean alignment score was 1.18 out of 3.

The two moderately aligned topics were both associated with Location and
contained relatively explicit geographical vocabulary such as Punta, Madrid,
Barcelona, Caribbean, Dominican, beaches, and tours. Other reduced topics
remained dominated by hotel brands, generic property terminology, attractions,
resorts, and amenities.

The LLM-based semantic assessment therefore supports the earlier quantitative
and qualitative findings. Hierarchical topic reduction reduced fragmentation
from 188 to 11 topics, but did not produce a topic structure that closely
corresponded to the Trip-MA aspect annotation scheme.

## 21. Overall Experimental Summary

This notebook evaluated whether C-Top2Vec can transfer from the original
benchmark setting to multi-aspect consumer reviews and whether its discovered
topic structure corresponds to human-defined TripAdvisor aspects.

On the 369 natural TripAdvisor reviews, C-Top2Vec completed successfully but
produced 188 topics. Inspection of these topics showed substantial fragmentation
and frequent organisation around hotel brands, destinations, resorts, amenities,
and other contextual information rather than direct correspondence with the
Trip-MA aspect taxonomy.

A second controlled dataset was therefore constructed using 100 synthetic
multi-aspect documents containing three known aspects and two known aspect
boundaries per document. This provided 200 ground-truth aspect transitions.
C-Top2Vec discovered six topics on this dataset, but topic transitions showed
limited correspondence with the known boundaries. Only 6 of 200 boundaries
were detected exactly. At a tolerance of ±5 whitespace tokens, 36 boundaries
were matched, giving precision of 0.350, recall of 0.180, and F1-score of 0.238.
Seventy-two of the 100 controlled documents contained no predicted topic
transition.

As a follow-up analysis, the 188 natural-review topics were hierarchically
reduced to 11 topics to match the size of the complete Trip-MA annotation
scheme. Although this reduced fragmentation, the resulting topics remained
dominated by hotel brands, destinations, resorts, attractions, amenities, and
general property vocabulary rather than clearly separating into the Trip-MA
aspects.

The LLM-as-Judge evaluation provided complementary semantic evidence. Across
the 11 reduced topics, the mean alignment score was 1.18 out of 3. Nine topics
were judged to have weak or mixed alignment and two had moderate alignment;
none received a strong alignment score.

Overall, the experiments show that C-Top2Vec can be executed successfully on
consumer-review data, but its latent topic structure does not naturally align
with the human-defined multi-aspect structure represented by Trip-MA. The
natural-review, controlled-boundary, hierarchical-reduction, and LLM-as-Judge
analyses provide complementary evidence for this conclusion.